# Auto-encodeurs débruiteurs et variationnels avec PyTorch Lightning

Le notebook précédent entraînait des auto-encodeurs à reconstruire leur entrée. Celui-ci change d'abord la tâche : retrouver une image propre à partir d'une version bruitée. Puis il change le modèle, pour pouvoir générer de nouvelles images. Les données sont celles de Fashion-MNIST : 60 000 images d'entraînement et 10 000 de test, 28 × 28 en niveaux de gris comme MNIST, mais de 10 sortes de vêtements et d'accessoires.

1. Corrompre les images
2. Un débruiteur dense
3. Un code sur-complet
4. Un débruiteur convolutif
5. Un auto-encodeur variationnel
6. Générer des images
7. L'effet de β

Chaque partie réutilise les données et les modèles des précédentes. Pas besoin de GPU : le notebook tourne en 20 à 30 minutes sur un processeur. Un GPU (`Exécution > Modifier le type d'exécution`) l'accélère.

## Installation et imports

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import torch
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

lightning.seed_everything(0)

## Chargement de Fashion-MNIST

In [ ]:
donnees_entrainement = torchvision.datasets.FashionMNIST(
  "donnees", train=True, download=True
)
donnees_test = torchvision.datasets.FashionMNIST("donnees", train=False, download=True)
noms_classes = [
  "T-shirt",
  "Pantalon",
  "Pull",
  "Robe",
  "Manteau",
  "Sandale",
  "Chemise",
  "Basket",
  "Sac",
  "Bottine",
]
dim_entree = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde
X_train = donnees_entrainement.data.reshape(-1, dim_entree).float() / 255
y_train = donnees_entrainement.targets
X_test = donnees_test.data.reshape(-1, dim_entree).float() / 255
y_test = donnees_test.targets
print(X_train.shape, X_test.shape)


def affiche_images(*rangees: torch.Tensor, titres: list | None = None) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rangees[0])
  _, axes = plt.subplots(
    len(rangees), n, figsize=(n * 1.2, len(rangees) * 1.3), squeeze=False
  )
  for rangee, images in zip(axes, rangees):
    for ax, image in zip(rangee, images):
      ax.imshow(image.reshape(28, 28), cmap="gray_r")
      ax.axis("off")
  for ax, titre in zip(axes[0], [] if titres is None else titres):
    ax.set_title(titre, fontsize=8)
  plt.show()


affiche_images(X_train[:10], titres=[noms_classes[y] for y in y_train[:10]])

## 1. Corrompre les images

Un débruiteur apprend à retrouver l'image propre à partir d'une version abîmée : ses exemples d'entraînement sont des paires (image bruitée, image propre). Fabriquez les images bruitées :

- `X_train_bruite` et `X_test_bruite` : `X_train` et `X_test`, plus un bruit gaussien de moyenne 0 et d'écart-type 0,5 tiré indépendamment pour chaque pixel ;
- ramenez ensuite chaque pixel dans [0, 1], comme ceux d'une vraie image.

La cellule après la solution affiche des images bruitées au-dessus des images propres.

**Documentation utile**

- [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)
- [`torch.Generator`](https://docs.pytorch.org/docs/stable/generated/torch.Generator.html)
- [`torch.clamp`](https://docs.pytorch.org/docs/stable/generated/torch.clamp.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `generateur = torch.Generator().manual_seed(0)`, puis `torch.randn(X_train.shape, generator=generateur)` : un bruit de moyenne 0 et d'écart-type 1, de la forme des images, un tirage par pixel ; multiplié par 0,5, son écart-type vaut 0,5
- `X.clamp(0, 1)` : les valeurs sous 0 deviennent 0, celles au-dessus de 1 deviennent 1
- vérification : `X_train_bruite.min()` vaut 0, `X_train_bruite.max()` vaut 1, et `X_train_bruite.shape` est celle de `X_train`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
generateur = torch.Generator().manual_seed(0)
X_train_bruite = (X_train + 0.5 * torch.randn(...)).clamp(..., ...)
X_test_bruite = ...
```

### Solution

In [ ]:
generateur = torch.Generator().manual_seed(0)
X_train_bruite = (
  X_train + 0.5 * torch.randn(X_train.shape, generator=generateur)
).clamp(0, 1)
X_test_bruite = (X_test + 0.5 * torch.randn(X_test.shape, generator=generateur)).clamp(
  0, 1
)

### Les images bruitées

In [ ]:
titres = [noms_classes[y] for y in y_test[:10]]
affiche_images(X_test_bruite[:10], X_test[:10], titres=titres)

## Les outils de l'entraînement

Ils sont fournis, et servent à tout le notebook :

- `AutoEncodeur`, un `LightningModule` qui enveloppe un réseau et une perte : son entrée est l'image bruitée, sa cible l'image propre ; il initialise les poids du réseau comme Keras, ce qui évite que des unités ReLU du code meurent ;
- `fabrique_chargeurs(X_bruite, X, X_test_bruite, X_test)`, qui rend les `DataLoader` d'entraînement et de validation des paires (image bruitée, image propre), par lots de 256 ; la validation se fait sur les images de test ;
- `entraine(modele, chargeurs, nb_epoques_max, nom)`, qui entraîne un `AutoEncodeur` et rend sa dernière perte de validation ;
- `predit(reseau, X)`, qui applique un réseau à toutes les images de `X`, lot par lot, sans calculer de gradient.

In [ ]:
class AutoEncodeur(lightning.LightningModule):
  """Enveloppe Lightning : reconstruit l'image propre à partir de l'image bruitée."""

  def __init__(
    self,
    modele: nn.Module,
    perte: nn.Module,
    taux_apprentissage: float = 1e-3,
    forme_entree: tuple[int, ...] = (dim_entree,),
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["modele", "perte"])
    self.modele = modele
    self.perte = perte
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, *forme_entree)
    # Initialisation de Glorot, celle de Keras : avec celle de PyTorch, la plupart
    # des unités ReLU d'un petit code meurent (nulles pour toutes les images)
    for couche in modele.modules():
      if isinstance(couche, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(couche.weight)
        nn.init.zeros_(couche.bias)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.modele(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    images_bruitees, images = batch
    perte = self.perte(self(images_bruitees), images)
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.taux_apprentissage)


def fabrique_chargeurs(
  X_bruite: torch.Tensor,
  X: torch.Tensor,
  X_test_bruite: torch.Tensor,
  X_test: torch.Tensor,
) -> tuple[DataLoader, DataLoader]:
  """Chargeurs d'entraînement et de validation des paires (image bruitée, image propre)."""
  chargeur_entrainement = DataLoader(
    TensorDataset(X_bruite, X), batch_size=256, shuffle=True
  )
  chargeur_val = DataLoader(TensorDataset(X_test_bruite, X_test), batch_size=256)
  return chargeur_entrainement, chargeur_val


def entraine(
  modele: AutoEncodeur,
  chargeurs: tuple[DataLoader, DataLoader],
  nb_epoques_max: int,
  nom: str,
) -> float:
  """Entraîne le modèle et rend sa dernière perte de validation."""
  trainer = lightning.Trainer(
    max_epochs=nb_epoques_max,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
    enable_model_summary=False,
  )
  trainer.fit(modele, *chargeurs)
  return trainer.callback_metrics["perte_val"].item()


@torch.no_grad()
def predit(modele: nn.Module, X: torch.Tensor, taille_lot: int = 256) -> torch.Tensor:
  """Applique un réseau à chaque image de X, lot par lot."""
  device = next(modele.parameters()).device
  mode_entrainement = modele.training
  modele.eval()
  predictions = [modele(batch.to(device)).cpu() for batch in X.split(taille_lot)]
  # Rend au réseau son mode : fit ne le remet plus en mode entraînement
  modele.train(mode_entrainement)
  return torch.cat(predictions)


chargeurs = fabrique_chargeurs(X_train_bruite, X_train, X_test_bruite, X_test)

## 2. Un débruiteur dense

Construisez `autoencodeur`, un débruiteur dense à une couche cachée, et `modele`, l'`AutoEncodeur` qui l'enveloppe :

- une couche `nn.Linear` de `dim_entree` vers `dim_code = 12` unités, suivie d'une ReLU : le code ;
- la couche de sortie, de `dim_code` vers `dim_entree`, suivie d'une activation adaptée à des pixels dans [0, 1] ;
- la perte `nn.MSELoss()`.

Affichez son résumé avec `ModelSummary`. La cellule après la solution l'entraîne sur les paires (image bruitée, image propre), puis affiche des images de test bruitées, leur version débruitée et l'image propre.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un seul `nn.Sequential` suffit : `nn.Linear`, `nn.ReLU()`, `nn.Linear`, puis l'activation de sortie
- la sortie : `nn.Sigmoid()`, qui rend des valeurs dans ]0, 1[, comme les pixels
- `print(ModelSummary(modele, max_depth=-1))` : 9,4 K paramètres pour la couche du code (784 × 12 + 12) et 10,2 K pour la sortie (12 × 784 + 784), 19,6 K en tout

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dim_code = 12
autoencodeur = nn.Sequential(nn.Linear(..., ...), nn.ReLU(), ..., ...)
modele = AutoEncodeur(autoencodeur, perte=...)
print(ModelSummary(modele, max_depth=-1))
```

### Solution

In [ ]:
dim_code = 12
autoencodeur = nn.Sequential(
  nn.Linear(dim_entree, dim_code),
  nn.ReLU(),
  nn.Linear(dim_code, dim_entree),
  nn.Sigmoid(),  # sortie dans ]0, 1[, comme les pixels
)
modele = AutoEncodeur(autoencodeur, perte=nn.MSELoss())
print(ModelSummary(modele, max_depth=-1))

### Entraînement

L'entrée est l'image bruitée, la cible l'image propre. En haut, les images bruitées, puis leur version débruitée, puis les images propres.

In [ ]:
perte_val = entraine(modele, chargeurs, nb_epoques_max=20, nom="dense_12")
print(f"Perte de validation (MSE) : {perte_val:.4f}")
X_test_debruite = predit(autoencodeur, X_test_bruite)
affiche_images(X_test_bruite[:10], X_test_debruite[:10], X_test[:10], titres=titres)

## 3. Un code sur-complet

Un code plus grand que l'entrée, sur-complet, laisse à un auto-encodeur ordinaire la place de recopier son entrée, bruit compris (slide « Code sous-complet ou sur-complet »). Et à un débruiteur ? Comparez-les, avec un code de `dim_code = 1024` unités, plus que les 784 pixels :

- écrivez `construit_autoencodeur_dense(dim_code)`, qui rend un réseau de la forme de la partie 2 (la couche du code, en ReLU, puis la sortie en sigmoïde) ;
- `ordinaire`, un auto-encodeur ordinaire, entraîné à reconstruire les images propres : ses `DataLoader` sont `fabrique_chargeurs(X_train, X_train, X_test, X_test)` ;
- `debruiteur`, un réseau de la même forme, entraîné en débruiteur sur `chargeurs`, les paires (image bruitée, image propre) ;
- 10 epochs chacun, avec la perte `nn.MSELoss()` ;
- appliquez-les tous deux à `X_test_bruite`, mesurez l'erreur quadratique moyenne de chacun par rapport aux images propres, et affichez les images bruitées, la sortie de `ordinaire`, celle de `debruiteur`, puis les images propres.

Lequel recopie le bruit ? Pourquoi l'autre ne le fait-il pas, alors qu'il en a la place ?

In [ ]:
def construit_autoencodeur_dense(dim_code: int) -> nn.Sequential: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `construit_autoencodeur_dense` rend le `nn.Sequential` de la partie 2, avec `dim_code` en paramètre ; `print(ModelSummary(AutoEncodeur(ordinaire, perte=nn.MSELoss()), max_depth=-1))` affiche 1,6 M paramètres (1 607 440)
- `chargeurs_ordinaires = fabrique_chargeurs(X_train, X_train, X_test, X_test)` : l'entrée et la cible sont toutes deux l'image propre
- `entraine(AutoEncodeur(ordinaire, perte=nn.MSELoss()), chargeurs_ordinaires, nb_epoques_max=10, nom="ordinaire_1024")` entraîne `ordinaire` sur place, et `entraine(..., chargeurs, ...)` le débruiteur
- `((predit(reseau, X_test_bruite) - X_test) ** 2).mean()` : l'erreur par rapport aux images propres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_autoencodeur_dense(dim_code: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(dim_entree, dim_code), nn.ReLU(), ..., ...)


dim_code = 1024
ordinaire, debruiteur = construit_autoencodeur_dense(dim_code), ...
chargeurs_ordinaires = fabrique_chargeurs(...)
entraine(
  AutoEncodeur(ordinaire, perte=nn.MSELoss()),
  chargeurs_ordinaires,
  nb_epoques_max=10,
  nom="ordinaire_1024",
)
entraine(
  AutoEncodeur(debruiteur, perte=...), ..., nb_epoques_max=..., nom="debruiteur_1024"
)

for nom, reseau in [("AE ordinaire", ordinaire), ("Débruiteur", debruiteur)]:
  erreur = ...
  print(f"{nom} : MSE {erreur:.4f} par rapport aux images propres")
affiche_images(X_test_bruite[:10], ..., ..., X_test[:10], titres=titres)
```

### Solution

In [ ]:
def construit_autoencodeur_dense(dim_code: int) -> nn.Sequential:
  return nn.Sequential(
    nn.Linear(dim_entree, dim_code),
    nn.ReLU(),
    nn.Linear(dim_code, dim_entree),
    nn.Sigmoid(),
  )


dim_code = 1024  # plus que les 784 pixels
ordinaire, debruiteur = (
  construit_autoencodeur_dense(dim_code),
  construit_autoencodeur_dense(dim_code),
)
# L'AE ordinaire : l'entrée et la cible sont l'image propre
chargeurs_ordinaires = fabrique_chargeurs(X_train, X_train, X_test, X_test)
entraine(
  AutoEncodeur(ordinaire, perte=nn.MSELoss()),
  chargeurs_ordinaires,
  nb_epoques_max=10,
  nom="ordinaire_1024",
)
# Le débruiteur : l'image bruitée en entrée, l'image propre en cible
entraine(
  AutoEncodeur(debruiteur, perte=nn.MSELoss()),
  chargeurs,
  nb_epoques_max=10,
  nom="debruiteur_1024",
)

for nom, reseau in [("AE ordinaire", ordinaire), ("Débruiteur", debruiteur)]:
  erreur = ((predit(reseau, X_test_bruite) - X_test) ** 2).mean()
  print(f"{nom} : MSE {erreur:.4f} par rapport aux images propres")
# En haut les images bruitées, puis l'AE ordinaire, le débruiteur, les images propres
affiche_images(
  X_test_bruite[:10],
  predit(ordinaire, X_test_bruite[:10]),
  predit(debruiteur, X_test_bruite[:10]),
  X_test[:10],
  titres=titres,
)

L'auto-encodeur ordinaire recopie l'essentiel du bruit : son code de 1 024 nombres lui laisse la place d'apprendre une fonction proche de l'identité, et, n'ayant vu que des images propres, rien ne lui a appris à effacer du bruit. Son erreur par rapport aux images propres, environ 0,05, est plus de trois fois celle du débruiteur, environ 0,015. Avec le même code sur-complet, le débruiteur rend des vêtements propres, plus nets qu'avec le code de 12 de la partie 2 (0,025) : sa cible est l'image propre, et chaque pixel de bruit recopié augmenterait sa perte. C'est la contrainte de la slide « Code sous-complet ou sur-complet » : un code sur-complet n'est utile que si la tâche interdit la copie, ici par une entrée bruitée.

## 4. Un débruiteur convolutif

Une couche dense traite chaque pixel à part ; une convolution exploite le voisinage de chaque pixel, ce qui convient mieux à une image. Les couches de convolution de PyTorch attendent des images 1 × 28 × 28, avec un axe pour les canaux juste après celui du lot (un seul canal en niveaux de gris, trois en couleur) : la cellule suivante remet les images dans cette forme, et prépare les `DataLoader` de ces images.

In [ ]:
X_train_img = X_train.reshape(-1, 1, 28, 28)
X_test_img = X_test.reshape(-1, 1, 28, 28)
X_train_bruite_img = X_train_bruite.reshape(-1, 1, 28, 28)
X_test_bruite_img = X_test_bruite.reshape(-1, 1, 28, 28)
print(X_train_img.shape)
chargeurs_images = fabrique_chargeurs(
  X_train_bruite_img, X_train_img, X_test_bruite_img, X_test_img
)

### Le modèle

Construisez un `encodeur` et un `decodeur` entièrement convolutifs, puis `autoencodeur`, qui les enchaîne sous ces deux noms, et `modele`, l'`AutoEncodeur` qui l'enveloppe. Toutes les `nn.Conv2d` ont des filtres 3 × 3 et `padding="same"` :

- `encodeur`, sur des images 1 × 28 × 28 : une `nn.Conv2d` de 16 filtres (ReLU) et un `nn.MaxPool2d` 2 × 2, puis une `nn.Conv2d` de 8 filtres (ReLU) et un `nn.MaxPool2d` 2 × 2. Le code est le tenseur 8 × 7 × 7 qui en sort ;
- `decodeur`, le chemin inverse : une `nn.Conv2d` de 8 filtres (ReLU) et un `nn.Upsample` de facteur 2, une `nn.Conv2d` de 16 filtres (ReLU) et un `nn.Upsample` de facteur 2, puis une dernière `nn.Conv2d` à 1 filtre suivie d'une sigmoïde, qui rend une image 1 × 28 × 28 ;
- `modele`, avec la perte des débruiteurs denses, `nn.MSELoss()`, et `forme_entree=(1, 28, 28)`, la forme d'une image, pour le résumé.

Vérifiez les formes avec `ModelSummary`, et comparez son nombre de paramètres à celui du débruiteur dense de la partie 3. Pourquoi ne pas finir le décodeur par un `nn.Upsample` ?

**Documentation utile**

- [`torch.nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [`torch.nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
- [`torch.nn.Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Conv2d(1, 16, 3, padding="same")` : 1 canal en entrée, 16 filtres 3 × 3 ; `padding="same"` garde la taille de l'image ; une petite fonction `cree_convolution(canaux_entree, canaux_sortie)` qui la renvoie évite de répéter ces arguments
- `nn.MaxPool2d(2)` divise la hauteur et la largeur par 2 (28 → 14 → 7), `nn.Upsample(scale_factor=2)` les multiplie par 2 en répétant chaque pixel (7 → 14 → 28)
- la dernière couche : `cree_convolution(16, 1)`, puis `nn.Sigmoid()` ; un `nn.Upsample` en fin de réseau rendrait des blocs de 2 × 2 pixels identiques, et une sortie sans sigmoïde pourrait sortir de [0, 1]
- `print(ModelSummary(modele, max_depth=2))` : la sortie de l'encodeur est `[1, 8, 7, 7]`, celle du décodeur `[1, 1, 28, 28]` (le 1 en tête est le lot de l'entrée d'exemple) ; 3,2 K paramètres en tout (3 217), contre 1,6 M pour le débruiteur dense de la partie 3 : chaque filtre 3 × 3 sert en tout point de l'image

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def cree_convolution(canaux_entree: int, canaux_sortie: int) -> nn.Conv2d:
  return nn.Conv2d(canaux_entree, canaux_sortie, 3, padding="same")


encodeur = nn.Sequential(
  cree_convolution(1, 16), nn.ReLU(), nn.MaxPool2d(2), ..., ..., ...
)
decodeur = nn.Sequential(
  cree_convolution(8, 8),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),
  ...,
  ...,
  ...,
  ...,
  ...,
)
autoencodeur = nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))
modele = AutoEncodeur(autoencodeur, perte=..., forme_entree=...)
print(ModelSummary(modele, max_depth=2))
```

#### Solution

In [ ]:
def cree_convolution(canaux_entree: int, canaux_sortie: int) -> nn.Conv2d:
  return nn.Conv2d(canaux_entree, canaux_sortie, 3, padding="same")


encodeur = nn.Sequential(
  cree_convolution(1, 16),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 16 × 14 × 14
  cree_convolution(16, 8),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 8 × 7 × 7, le code
)
decodeur = nn.Sequential(
  cree_convolution(8, 8),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),  # 8 × 14 × 14
  cree_convolution(8, 16),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),  # 16 × 28 × 28
  # Lisse les blocs de l'Upsample, puis sortie dans ]0, 1[
  cree_convolution(16, 1),
  nn.Sigmoid(),
)
autoencodeur = nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))
modele = AutoEncodeur(autoencodeur, perte=nn.MSELoss(), forme_entree=(1, 28, 28))
print(ModelSummary(modele, max_depth=2))

### Entraînement

La même perte que les débruiteurs denses, l'erreur quadratique moyenne, mais 5 epochs seulement, contre 20 pour celui de la partie 2 et 10 pour ceux de la partie 3 : la comparaison avantage les denses.

In [ ]:
# 5 epochs, contre 20 et 10 pour les débruiteurs denses ; même perte, la MSE
perte_val = entraine(modele, chargeurs_images, nb_epoques_max=5, nom="conv")
print(f"Perte de validation (MSE) : {perte_val:.4f}")
X_test_debruite = predit(autoencodeur, X_test_bruite_img)
affiche_images(X_test_bruite[:10], X_test_debruite[:10], X_test[:10], titres=titres)

### Sur des images non bruitées

Le débruiteur n'a vu que des entrées bruitées. Que rend-il d'une image propre ?

In [ ]:
X_test_reconstruit = predit(autoencodeur, X_test_img[:10])
affiche_images(X_test[:10], X_test_reconstruit, titres=titres)

Sur une image propre, le débruiteur rend la forme du vêtement, mais lissée : il efface les détails fins, comme le logo du pull, les carreaux de la chemise ou les brides de la sandale, et force le contraste. Il a appris à garder ce qui ressemble à un vêtement et à effacer le reste ; un détail fin, à ses yeux, ressemble à du bruit. Son erreur quadratique moyenne sur les images bruitées, environ 0,021 après 5 epochs seulement, se place entre celles des débruiteurs denses à code de 12 (0,025) et de 1 024 (0,015), avec 3 217 paramètres au lieu de 19 612 et 1 607 440 : chaque filtre 3 × 3 sert en tout point de l'image, quand chaque poids d'une couche dense ne relie qu'un pixel à une unité.

## 5. Un auto-encodeur variationnel

Le notebook précédent l'a montré : décoder un code tiré au hasard ne donne pas une image vraisemblable, car les codes d'un auto-encodeur n'occupent qu'une petite partie de l'espace latent, sans forme connue. Un auto-encodeur variationnel (VAE) encode chaque image en une loi, $\mathcal{N}(\mu, \sigma^2)$, et une perte KL pousse ces lois vers $\mathcal{N}(0, I)$ : on peut alors tirer un code dans $\mathcal{N}(0, I)$ et le décoder.

Le réseau du VAE est celui de la slide « Un VAE en PyTorch » : il est fourni plus bas, avec son enveloppe Lightning. Il lui manque les deux pièces propres au VAE, à écrire d'abord.

### Le tirage et la KL

Écrivez, d'après les slides « Astuce de reparamétrisation » et « Perte d'un VAE » :

- `Echantillonnage`, un `nn.Module` dont `forward(z_moyenne, z_log_var)` reçoit $\mu$ et $\log \sigma^2$, et tire un code $z \sim \mathcal{N}(\mu, \sigma^2)$ par l'astuce de reparamétrisation, pour que le gradient de la perte atteigne $\mu$ et $\sigma$ ;
- `divergence_kl(z_moyenne, z_log_var)`, qui rend, pour chaque ligne du lot, la KL entre $\mathcal{N}(\mu, \sigma^2)$ et $\mathcal{N}(0, I)$ : un tenseur d'une valeur par image.

La cellule après la solution les vérifie sur des exemples calculables à la main.

**Documentation utile**

- [`torch.randn_like`](https://docs.pytorch.org/docs/stable/generated/torch.randn_like.html)
- [`torch.exp`](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)
- [`torch.sum`](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

In [ ]:
class Echantillonnage(nn.Module):
  def forward(self, z_moyenne: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    """Un code tiré dans N(z_moyenne, exp(z_log_var)), dérivable en ses deux entrées."""
    ...


def divergence_kl(z_moyenne: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
  """La KL entre N(z_moyenne, exp(z_log_var)) et N(0, I), une valeur par ligne."""
  ...


# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- le tirage : $z = \mu + \sigma \varepsilon$, avec $\varepsilon \sim \mathcal{N}(0, I)$ tiré à part ; `torch.randn_like(z_moyenne)` tire $\varepsilon$ de la forme de `z_moyenne`
- $\sigma = \exp\left(\frac{1}{2} \log \sigma^2\right)$ : `torch.exp(0.5 * z_log_var)`
- la KL : $-\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$, la somme portant sur les dimensions du code, `dim=1` ; $\sigma_j^2$ est `z_log_var.exp()`
- vérification à la main : pour $\mu = (0, 1)$ et $\sigma = (1, 1)$, soit `z_log_var` nul, seul le terme $\mu_2^2 = 1$ compte, et la KL vaut 0,5

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class Echantillonnage(nn.Module):
  def forward(self, z_moyenne: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    eps = ...
    return z_moyenne + ... * eps


def divergence_kl(z_moyenne: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
  return -0.5 * torch.sum(..., dim=1)
```

#### Solution

In [ ]:
class Echantillonnage(nn.Module):
  def forward(self, z_moyenne: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    """Un code tiré dans N(z_moyenne, exp(z_log_var)), dérivable en ses deux entrées."""
    eps = torch.randn_like(z_moyenne)  # ε ~ N(0, I), tiré à part
    return z_moyenne + torch.exp(0.5 * z_log_var) * eps  # z = μ + σ·ε


def divergence_kl(z_moyenne: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
  """La KL entre N(z_moyenne, exp(z_log_var)) et N(0, I), une valeur par ligne."""
  return -0.5 * torch.sum(1 + z_log_var - z_moyenne**2 - z_log_var.exp(), dim=1)

### Vérification

La KL de $\mathcal{N}((0, 1), I)$ vaut 0,5 ; tirés dans $\mathcal{N}(0, 4)$, soit $\log \sigma^2 = \log 4$, des codes ont une moyenne proche de 0 et un écart-type proche de 2 sur chaque dimension.

In [ ]:
print("KL :", divergence_kl(torch.tensor([[0.0, 1.0]]), torch.zeros(1, 2)))
z = Echantillonnage()(torch.zeros(10_000, 2), torch.full((10_000, 2), 4.0).log())
print("Moyenne :", z.mean(dim=0), "écart-type :", z.std(dim=0))

### Le modèle

Ils sont fournis :

- `VAE`, un `AutoEncodeur` pour un réseau qui rend la reconstruction, `z_moyenne` et `z_log_var` : sa perte est celle de la reconstruction, plus `beta` fois la moyenne de `divergence_kl` sur le lot (slide « Échantillonnage et perte en PyTorch ») ;
- `AutoEncodeurVariationnel`, le réseau de la slide « Un VAE en PyTorch », qui tire son code avec `Echantillonnage` ;
- `reseau`, un `AutoEncodeurVariationnel` à `dim_latente = 2` dimensions, `modele`, le `VAE` qui l'enveloppe, avec la perte `nn.BCELoss()`, et `chargeurs_vae`, ses `DataLoader` : l'entrée et la cible sont toutes deux l'image propre.

Le résumé affiche 404 K paramètres en tout (404 244), presque tous dans la couche cachée de l'encodeur et dans la sortie du décodeur.

In [ ]:
class VAE(AutoEncodeur):
  """Enveloppe Lightning d'un VAE : perte de reconstruction, plus beta fois la KL."""

  def __init__(
    self,
    modele: nn.Module,
    perte: nn.Module,
    beta: float = 1 / 784,
    taux_apprentissage: float = 1e-3,
  ) -> None:
    super().__init__(modele, perte, taux_apprentissage)
    self.beta = beta  # le poids de la KL

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    entrees, images = batch
    reconstructions, z_moyenne, z_log_var = self(entrees)
    kl = divergence_kl(z_moyenne, z_log_var)
    perte = self.perte(reconstructions, images) + self.beta * kl.mean()
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    return perte


class AutoEncodeurVariationnel(nn.Module):
  """Le réseau du VAE : un encodeur à deux sorties, le tirage, un décodeur."""

  def __init__(self, dim_latente: int) -> None:
    super().__init__()
    self.couche_cachee = nn.Sequential(nn.Linear(dim_entree, 256), nn.ReLU())
    self.z_moyenne = nn.Linear(256, dim_latente)
    self.z_log_var = nn.Linear(256, dim_latente)  # log σ², un réel
    self.echantillonnage = Echantillonnage()
    self.decodeur = nn.Sequential(
      nn.Linear(dim_latente, 256),
      nn.ReLU(),
      nn.Linear(256, dim_entree),
      nn.Sigmoid(),
    )

  def encode(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """z_moyenne et z_log_var d'un lot d'images."""
    h = self.couche_cachee(images)
    return self.z_moyenne(h), self.z_log_var(h)

  def forward(
    self, images: torch.Tensor
  ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Les reconstructions, z_moyenne et z_log_var d'un lot d'images."""
    z_moyenne, z_log_var = self.encode(images)
    z = self.echantillonnage(z_moyenne, z_log_var)  # z = μ + σ·ε
    return self.decodeur(z), z_moyenne, z_log_var


# beta = 1 / 784 : la perte BCELoss est une moyenne sur les 784 pixels,
# alors que la KL est une somme sur les dimensions du code
chargeurs_vae = fabrique_chargeurs(X_train, X_train, X_test, X_test)
dim_latente = 2
reseau = AutoEncodeurVariationnel(dim_latente)
modele = VAE(reseau, perte=nn.BCELoss())  # + KL
print(ModelSummary(modele, max_depth=2))

### Entraînement et espace latent

La cible est l'image propre elle-même. Chaque point du nuage est le $\mu$ d'une image de test, coloré selon sa classe, que trace `affiche_latent`, une fonction qui servira encore dans la partie 7.

In [ ]:
def affiche_latent(reseau: AutoEncodeurVariationnel, ax: plt.Axes) -> None:
  # Le nuage des μ des images de test, coloré selon leur classe
  with torch.no_grad():
    moyennes_test, _ = reseau.encode(X_test)
  for etiquette, nom in enumerate(noms_classes):
    points = moyennes_test[y_test == etiquette]
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=nom)
  ax.set_xlabel("$z_1$")
  ax.set_ylabel("$z_2$")


entraine(modele, chargeurs_vae, nb_epoques_max=20, nom="vae")
_, ax = plt.subplots(figsize=(7, 6))
affiche_latent(reseau, ax)
ax.legend(markerscale=5, fontsize=8)
plt.show()

Les μ des images de test restent presque tous entre -3 et 3 sur chaque axe, autour de l'origine : la KL tire chaque loi vers $\mathcal{N}(0, I)$. Chaque classe occupe sa région. Les chaussures (sandales, baskets, bottines) se regroupent d'un côté du nuage, séparées du reste par une bande presque vide ; les sacs ont leur région, les pantalons aussi, à côté des robes et des T-shirts. Les autres hauts (pulls, manteaux, chemises) se chevauchent : ils se ressemblent, et deux dimensions ne suffisent pas à les séparer.

## 6. Générer des images

Comme à la fin de la slide « Un VAE en PyTorch », la cellule suivante tire 16 codes dans $\mathcal{N}(0, I)$ et les décode avec `reseau.decodeur`. Les images ressemblent-elles à des vêtements ? Comparez-les aux images décodées depuis du bruit dans le notebook précédent.

In [ ]:
# z ~ N(0, I)
z_nouveau = torch.randn(16, dim_latente, generator=torch.Generator().manual_seed(0))
generees = predit(reseau.decodeur, z_nouveau)
affiche_images(generees)

La plupart ressemblent à des vêtements : des hauts, des pantalons, des chaussures, flous comme toutes les images d'un VAE à deux dimensions. Quelques-unes, pâles, mélangent deux sortes de vêtements : leur code est tombé entre deux régions, où le décodeur hésite. Dans le notebook précédent, un code tiré au hasard ne donnait que des taches : ici, la KL a placé les codes des images d'entraînement là où $\mathcal{N}(0, I)$ tire les siens, et le décodeur sait les décoder. Un espace latent plus grand donnerait des images plus nettes, mais on ne pourrait plus en dessiner la carte.

### La carte de l'espace latent

Avec 2 dimensions, on peut décoder une grille régulière de codes, de -2 à 2 sur chaque axe, là où $\mathcal{N}(0, I)$ tire presque tous ses points : chaque case montre ce que le décodeur rend en ce point. Comparez avec le nuage de la partie 5.

In [ ]:
n = 12
grille = torch.linspace(-2, 2, n).tolist()
# z_2 décroît de haut en bas, comme l'axe vertical du nuage
codes = torch.tensor([[z1, z2] for z2 in grille[::-1] for z1 in grille])
images = predit(reseau.decodeur, codes).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(images.permute(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray_r")
plt.axis("off")
plt.show()

## 7. L'effet de β

La slide « Perte d'un VAE » l'annonce : sans la KL, on retombe sur un auto-encodeur ; avec un $\beta$ plus grand, l'espace latent est plus régulier, mais les images plus floues. Vérifiez-le :

- entraînez deux autres `AutoEncodeurVariationnel` à `dim_latente` dimensions, enveloppés chacun dans un `VAE`, pendant 20 epochs comme celui de la partie 5 : l'un avec `beta=0`, l'autre avec un $\beta$ dix fois plus grand que celui de la partie 5, `10 / 784` ;
- tracez côte à côte, avec `affiche_latent`, les nuages des $\mu$ des trois VAE ($\beta$ = 0, 1/784 et 10/784) ;
- décodez, avec chacun, les 16 codes `z_nouveau` de la partie 6, et affichez les trois rangées d'images.

Que deviennent le nuage et les images sans la KL ? Avec un $\beta$ dix fois plus grand ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `VAE(reseau_beta, perte=nn.BCELoss(), beta=0.0)`, puis `entraine(..., chargeurs_vae, nb_epoques_max=20, nom="vae_beta_0")` : `reseau_beta` est entraîné sur place
- un dictionnaire `{"0": ..., "1/784": reseau, "10/784": ...}` des trois réseaux garde le VAE de la partie 5 et donne les titres des figures
- `_, axes = plt.subplots(1, 3, figsize=(18, 5))`, puis, pour chaque réseau, `affiche_latent(reseau_beta, ax)` et `ax.set_title(...)` ; chaque nuage a ses propres axes : lisez leurs graduations
- `predit(reseau_beta.decodeur, z_nouveau)` : 16 images, que `affiche_images` affiche
- les deux entraînements prennent quelques minutes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
reseaux = {
  "0": AutoEncodeurVariationnel(dim_latente),
  "1/784": reseau,
  "10/784": ...,
}
for nom_beta, beta in [("0", 0.0), ("10/784", ...)]:
  entraine(
    VAE(reseaux[nom_beta], perte=..., beta=...),
    chargeurs_vae,
    nb_epoques_max=...,
    nom=f"vae_beta_{beta:.4f}",
  )

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nom_beta, reseau_beta) in zip(axes, reseaux.items()):
  ...
  ax.set_title(f"β = {nom_beta}")
plt.show()

for nom_beta, reseau_beta in reseaux.items():
  print(f"β = {nom_beta}")
  affiche_images(...)
```

### Solution

In [ ]:
reseaux = {
  "0": AutoEncodeurVariationnel(dim_latente),
  "1/784": reseau,  # le VAE de la partie 5
  "10/784": AutoEncodeurVariationnel(dim_latente),
}
for nom_beta, beta in [("0", 0.0), ("10/784", 10 / 784)]:
  entraine(
    VAE(reseaux[nom_beta], perte=nn.BCELoss(), beta=beta),
    chargeurs_vae,
    nb_epoques_max=20,
    nom=f"vae_beta_{beta:.4f}",
  )

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nom_beta, reseau_beta) in zip(axes, reseaux.items()):
  affiche_latent(reseau_beta, ax)
  ax.set_title(f"β = {nom_beta}")
axes[0].legend(markerscale=5, fontsize=8)
plt.show()

for nom_beta, reseau_beta in reseaux.items():
  print(f"β = {nom_beta}")
  affiche_images(predit(reseau_beta.decodeur, z_nouveau))

Sans la KL ($\beta = 0$), le VAE redevient un auto-encodeur : rien ne retient les $\mu$, qui s'étalent sur des dizaines d'unités, et chaque classe part dans sa propre direction, loin de l'origine. Les 16 codes tirés dans $\mathcal{N}(0, I)$ tombent alors tous dans une petite zone centrale où presque aucune image d'entraînement n'a son code : le décodeur n'en tire que des taches. Avec un $\beta$ dix fois plus grand (le $\beta = 10$ de la slide, puisque la perte de reconstruction est ici une moyenne sur les 784 pixels), le nuage est plus serré, entre -2 et 2 environ, et les classes s'y touchent davantage ; les images générées sont toutes des vêtements, sans les mélanges de deux sortes que donne parfois $\beta = 1/784$, mais plus lisses, sans détail. Le poids de la KL règle ce compromis : un espace latent où tout code tiré se décode, contre des reconstructions fidèles.